In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.path import RAW_DATA_FILE

In [ ]:
train_df = pd.read_csv(RAW_DATA_FILE)
train_df.head()

,text,label,prompt_name,source,RDizzl3_seven
0,Phones\n\nModern humans today are always on th...,0,Phones and driving,persuade_corpus,False
1,This essay will explain if drivers should or s...,0,Phones and driving,persuade_corpus,False
2,Driving while the use of cellular devices\n\nT...,0,Phones and driving,persuade_corpus,False
3,Phones & Driving\n\nDrivers should not be able...,0,Phones and driving,persuade_corpus,False
4,Cell Phone Operation While Driving\n\nThe abil...,0,Phones and driving,persuade_corpus,False


Xử lý null & mising data, trùng lập


In [3]:
# Loại bỏ text rỗng, chỉ có khoảng trắng, hoặc quá ngắn để có ý nghĩa (< 20 từ)
train_df['text'] = train_df['text'].astype(str).str.strip()
train_df = train_df[train_df['text'].str.len() > 0]

train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))
too_short = train_df[train_df['word_count'] < 20]
print(f"Số dòng quá ngắn (<20 từ): {len(too_short)}")

train_df = train_df[train_df['word_count'] >= 20].reset_index(drop=True)
print("Shape sau khi loại text quá ngắn:", train_df.shape)

Số dòng quá ngắn (<20 từ): 2
Shape sau khi loại text quá ngắn: (44866, 6)


In [4]:
# group-wise, không lọc global trước đó nữa

def remove_outliers_iqr(df, col, group_col, k=3):
    df_list = []
    for label_value, group in df.groupby(group_col):
        Q1 = group[col].quantile(0.25)
        Q3 = group[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = max(0, Q1 - k * IQR)
        upper = Q3 + k * IQR
        filtered = group[(group[col] >= lower) & (group[col] <= upper)]
        print(f"Label {label_value}: upper={upper:.0f} → loại {len(group)-len(filtered)}/{len(group)} dòng")
        df_list.append(filtered)
    return pd.concat(df_list).reset_index(drop=True)

print("Shape sau khi lọc outlier (1 lần duy nhất, group-wise):", train_df.shape)

Shape sau khi lọc outlier (1 lần duy nhất, group-wise): (44866, 6)


In [5]:
#kiểm tra
print("Shape hiện tại:", train_df.shape)
print("Phân bố nhãn hiện tại:")
print(train_df['label'].value_counts())

Shape hiện tại: (44866, 6)
Phân bố nhãn hiện tại:
label
0    27371
1    17495
Name: count, dtype: int64


In [6]:
#chuẩn hóa
import re

def clean_text(text):
    text = text.lower()                                  # chuyển chữ thường
    text = re.sub(r'http\S+|www\S+', '', text)            # xóa URL
    text = re.sub(r'\S+@\S+', '', text)                   # xóa email
    text = re.sub(r'<.*?>', '', text)                      # xóa HTML tag nếu có
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?\']', ' ', text)  # giữ lại chữ, số, dấu câu cơ bản
    text = re.sub(r'\s+', ' ', text).strip()               # xóa khoảng trắng thừa
    return text

train_df['text_clean'] = train_df['text'].apply(clean_text)

# Cập nhật lại word_count sau khi làm sạch
train_df['word_count_clean'] = train_df['text_clean'].apply(lambda x: len(x.split()))

train_df[['text', 'text_clean']].head(10)

,text,text_clean
0,Phones\n\nModern humans today are always on th...,phones modern humans today are always on their...
1,This essay will explain if drivers should or s...,this essay will explain if drivers should or s...
2,Driving while the use of cellular devices\n\nT...,driving while the use of cellular devices toda...
3,Phones & Driving\n\nDrivers should not be able...,phones driving drivers should not be able to u...
4,Cell Phone Operation While Driving\n\nThe abil...,cell phone operation while driving the ability...
5,Cell phone use should not be legal while drivi...,cell phone use should not be legal while drivi...
6,Phones and Driving\n\nDriving is a good way to...,phones and driving driving is a good way to ge...
7,PHONES AND DRIVING\n\nIn this world in which w...,phones and driving in this world in which we l...
8,People are debating whether if drivers should ...,people are debating whether if drivers should ...
9,Texting and driving\n\nOver half of drivers in...,texting and driving over half of drivers in to...


In [7]:
#xử lý trùng lập
print("Số dòng trùng lặp hoàn toàn:", train_df.duplicated().sum())
print("Số dòng trùng lặp theo text_clean:", train_df.duplicated(subset=['text_clean']).sum())

before = train_df.shape[0]
train_df = train_df.drop_duplicates(subset=['text_clean']).reset_index(drop=True)
after = train_df.shape[0]
print(f"Đã xóa {before - after} dòng trùng lặp. Shape hiện tại: {train_df.shape}")

Số dòng trùng lặp hoàn toàn: 1
Số dòng trùng lặp theo text_clean: 12
Đã xóa 12 dòng trùng lặp. Shape hiện tại: (44854, 8)


In [8]:
#kiểm tra
print("Shape hiện tại:", train_df.shape)
print("Phân bố nhãn hiện tại:")
print(train_df['label'].value_counts())

Shape hiện tại: (44854, 8)
Phân bố nhãn hiện tại:
label
0    27359
1    17495
Name: count, dtype: int64


In [9]:
train_df.head()

,text,label,prompt_name,source,RDizzl3_seven,word_count,text_clean,word_count_clean
0,Phones\n\nModern humans today are always on th...,0,Phones and driving,persuade_corpus,False,379,phones modern humans today are always on their...,379
1,This essay will explain if drivers should or s...,0,Phones and driving,persuade_corpus,False,366,this essay will explain if drivers should or s...,366
2,Driving while the use of cellular devices\n\nT...,0,Phones and driving,persuade_corpus,False,178,driving while the use of cellular devices toda...,178
3,Phones & Driving\n\nDrivers should not be able...,0,Phones and driving,persuade_corpus,False,212,phones driving drivers should not be able to u...,211
4,Cell Phone Operation While Driving\n\nThe abil...,0,Phones and driving,persuade_corpus,False,332,cell phone operation while driving the ability...,332


In [10]:
print(train_df.columns.tolist())

['text', 'label', 'prompt_name', 'source', 'RDizzl3_seven', 'word_count', 'text_clean', 'word_count_clean']


In [11]:
train_df = train_df.drop(columns=['text'])
train_df = train_df.drop(columns=['RDizzl3_seven'])
train_df = train_df.drop(columns=['word_count'])
print(train_df.columns.tolist())

['label', 'prompt_name', 'source', 'text_clean', 'word_count_clean']


In [12]:
train_df.head()

,label,prompt_name,source,text_clean,word_count_clean
0,0,Phones and driving,persuade_corpus,phones modern humans today are always on their...,379
1,0,Phones and driving,persuade_corpus,this essay will explain if drivers should or s...,366
2,0,Phones and driving,persuade_corpus,driving while the use of cellular devices toda...,178
3,0,Phones and driving,persuade_corpus,phones driving drivers should not be able to u...,211
4,0,Phones and driving,persuade_corpus,cell phone operation while driving the ability...,332


In [13]:
assert train_df['text_clean'].isnull().sum() == 0, "Vẫn còn missing text!"
assert (train_df['text_clean'].str.strip() == '').sum() == 0, "Vẫn còn text rỗng!"
assert train_df.duplicated(subset=['text_clean']).sum() == 0, "Vẫn còn trùng lặp!"
assert train_df['label'].nunique() == 2, "Thiếu 1 trong 2 nhãn!"
assert train_df.isnull().sum().sum() == 0, "Vẫn còn missing ở cột khác!"

print("=== Phân bố nhãn SAU khi clean ===")
print(train_df['label'].value_counts())
print(train_df['label'].value_counts(normalize=True) * 100)

print("\n=== Thống kê word_count_clean SAU khi clean ===")
print(train_df.groupby('label')['word_count_clean'].describe())

print("\n✅ Dữ liệu đã qua kiểm tra đầy đủ, sẵn sàng lưu.")

=== Phân bố nhãn SAU khi clean ===
label
0    27359
1    17495
Name: count, dtype: int64
label
0    60.995675
1    39.004325
Name: proportion, dtype: float64

=== Thống kê word_count_clean SAU khi clean ===
         count        mean         std    min    25%    50%    75%     max
label                                                                     
0      27359.0  420.126138  190.545086  143.0  275.0  385.0  521.0  1656.0
1      17495.0  331.267276   95.004549   42.0  276.0  328.0  388.0   806.0

✅ Dữ liệu đã qua kiểm tra đầy đủ, sẵn sàng lưu.


In [14]:
from src.path import PROCESSED_DATA_FILE

output_path = PROCESSED_DATA_FILE
train_df.to_csv(output_path, index=False)

print(f"Đã lưu dữ liệu sạch tại: {output_path}")
print(f"Shape cuối cùng: {train_df.shape}")
print(f"Các cột: {train_df.columns.tolist()}")

Đã lưu dữ liệu sạch tại: H:\FPT\4\DAP\Project-19\AI-Generated-Text-Essay-Detection\data\processed\processed_data.csv
Shape cuối cùng: (44854, 5)
Các cột: ['label', 'prompt_name', 'source', 'text_clean', 'word_count_clean']
